In [8]:
import xarray as xr
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point
import cartopy.feature as cf
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.colors as mcolors
from pathlib import Path
%run utils/utils_helper_funcs.ipynb

Exception: File `'utils/utils_helper_funcs.ipynb'` not found.

In [1]:
def whole_spatial_diff_plot(ds1, ds2, aerosol, time_range=None):
    # Handle dates
    try:
        len(ds1['time'])==len(ds2['time'])
    except:
        raise KeyError('Both datasets must have the same length time coordinate')
        
    if time_range != None:
        start = str(time_range[0])
        end = str(time_range[1])
        ds1 = ds1.sel(time=slice(start, end))
        ds2 = ds2.sel(time=slice(start, end))
    else:
            start = str((ds1['time'].values)[0])[:4]
            end = str((ds1['time'].values)[len(ds1['time'])-1])[:4]

    ds_diff = ds1.copy()
    if aerosol=='ozone':
        ds_diff['O3'] = ds1.O3 - ds2.O3
    elif aerosol=='PM25':
        ds_diff['PM25'] = ds1.PM25 - ds2.PM25
    else:
        raise ValueError('Please choose either ozone or PM25')

    # Change longitude for plotting
    if max(ds_diff.lon)>350:
        ds_diff = ds_diff.assign_coords(lon=(((ds_diff.lon + 180) % 360) - 180)).sortby('lon')
    ds_diff = ds_diff.mean(dim=['time','lev'])
    var_plot = ds_diff.O3
    display(var_plot)
    
    # Create the plot
    proj = ccrs.PlateCarree()
    fig = plt.figure(figsize=(10,12))
    ax = plt.axes(projection = proj)
    ax.coastlines()
    ax.add_feature(cf.BORDERS.with_scale('50m'), edgecolor = [.3,.3,.3], linewidth = 0.5)

    data_cyc, lon_cyc = add_cyclic_point(var_plot, coord = var_plot.lon)
    o = ax.contourf(lon_cyc, var_plot.lat, data_cyc, transform=ccrs.PlateCarree(), cmap=matplotlib.cm.YlGnBu_r)
    cb = plt.colorbar(o, extendfrac = 'auto', shrink=0.25)

    a_title = aerosol.capitalize()
    cb.set_label(f'{a_title} Concentration (mol/mol)')
    plt.title(f'Global Atmospheric {a_title} Difference, {start} - {end}')

In [18]:
def sfc_spatial_diff_plot(ds1, ds2, aerosol, time_range=None):
    # Handle dates
    try:
        len(ds1['time'])==len(ds2['time'])
    except:
        raise KeyError('Both datasets must have the same length time coordinate')
        
    if time_range != None:
        start = str(time_range[0])
        end = str(time_range[1])
        ds1 = ds1.sel(time=slice(start, end))
        ds2 = ds2.sel(time=slice(start, end))
    else:
            start = str((ds1['time'].values)[0])[:4]
            end = str((ds1['time'].values)[len(ds1['time'])-1])[:4]

    ds_diff = ds1.copy()
    if aerosol=='ozone':
        ds_diff['O3'] = ds1.O3 - ds2.O3
    elif aerosol=='PM25':
        ds_diff['PM25'] = ds1.PM25 - ds2.PM25
    else:
        raise ValueError('Please choose either ozone or PM25')

    # Change longitude for plotting
    if max(ds_diff.lon)>350:
        ds_diff = ds_diff.assign_coords(lon=(((ds_diff.lon + 180) % 360) - 180)).sortby('lon')
    ds_diff = ds_diff.mean(dim=['time'])
    var_plot = ds_diff.O3
    var_plot = var_plot.sel(lev=max(var_plot.lev.values))

    # Create the plot
    proj = ccrs.PlateCarree()
    fig = plt.figure(figsize=(10,12))
    ax = plt.axes(projection = proj)
    ax.coastlines()
    ax.add_feature(cf.BORDERS.with_scale('50m'), edgecolor = [.3,.3,.3], linewidth = 0.5)

    norm = mcolors.TwoSlopeNorm(vmin=-0.4e-8, vcenter=0, vmax=3.2e-8)
    data_cyc, lon_cyc = add_cyclic_point(var_plot, coord = var_plot.lon)
    o = ax.contourf(lon_cyc, var_plot.lat, data_cyc, transform=ccrs.PlateCarree(), cmap=matplotlib.cm.BrBG, norm=norm)
    cb = plt.colorbar(o, extendfrac = 'auto', shrink=0.25)

    a_title = aerosol.capitalize()
    cb.set_label(f'{a_title} Concentration (mol/mol)')
    plt.title(f'Global Surface {a_title} Difference, {start} - {end}')